# F03 — Synthetic ANES 2024 respondents, stage S2 (Study 3)

Generates synthetic survey respondents with open-weight chat models for the survey study.
Each synthetic respondent receives a persona drawn from the background variables of a real
ANES 2024 respondent and answers four feeling thermometers (the outcomes) and the eight
like/dislike questions about the candidates and parties (the text the detector sees).

**Runtime: GPU (T4).** Run the notebook once per model (choose it in cell 4). Each model
produces 1,600 respondents: 4 conditions (persona with or without party identification and
ideology x plain or human-like writing style) x 400. About 40-70 minutes per model. All
replies are cached on Google Drive after every 64 respondents, so after a disconnection
simply run all cells again: finished work is not repeated.

**Input:** two ANES 2024 files copied into the Drive folder of cell 2, unchanged: the main
data file `anes_timeseries_2024_csv_20260519.zip` and the open-ended answers
`anes_timeseries_2024_redactedopenends_excel_20250923.xlsx`. Personas are drawn from the
4,926 respondents of the human pool (same rules as `d2vi.data.anes`).

**Output (in the same Drive folder):** `anes_synth_<model>.parquet` and its manifest.
Send these two files back. Do not upload ANES files to the public repository: ANES data
stay on your Drive. Synthetic respondents are used only to evaluate estimators.


In [ ]:
# Fresh runtime recommended (Runtime > Disconnect and delete runtime) if d2vi was installed before.
!pip -q install -U transformers accelerate bitsandbytes openpyxl
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPU:", torch.cuda.is_available())
assert d2vi.__version__ >= "2.0.0.dev3", "old d2vi still loaded: restart the runtime and run all"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, glob
WORK = "/content/drive/MyDrive/d2vi_F03"
os.makedirs(WORK, exist_ok=True)
print("Put the ANES 2024 CSV zip and the open-ends xlsx in:", WORK)
print(sorted(os.listdir(WORK)))


In [ ]:
from d2vi.data.anes import ANES2024 as ANESFiles
from d2vi.synthetic import ANES2024_CODEBOOK
csvs = sorted(glob.glob(f"{WORK}/anes_timeseries_2024_csv*.zip")) or sorted(glob.glob(f"{WORK}/anes_timeseries_2024_csv*.csv"))
xlsx = sorted(glob.glob(f"{WORK}/anes_timeseries_2024_redactedopenends*.xlsx"))
assert csvs and xlsx, "ANES main data (zip or csv) and redacted open-ends (xlsx) must both be in " + WORK
src = csvs[-1]
anes = ANESFiles(src, xlsx[-1]).human_pool()   # the eligible human records
print(src, "| human pool:", anes.shape, anes["mode"].value_counts().to_dict())
assert len(anes) == 4926, "pool size differs from the one built in the repository (4,926)"
ANES2024_CODEBOOK.validate(anes)   # stops if a persona variable or code is not as expected
print("persona variables OK")


In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"  #@param ["Qwen/Qwen2.5-7B-Instruct", "allenai/OLMo-2-1124-7B-Instruct", "microsoft/Phi-3.5-mini-instruct"]
PER_CONDITION = 400  #@param {type:"integer"}
from d2vi.synthetic.generator import HFChatGenerator, SyntheticSurvey, GENERATORS
gen = HFChatGenerator(MODEL, batch_size=16, max_new_tokens=450)
SLUG = MODEL.split("/")[-1]
print(gen.settings)


In [ ]:
# Smoke test: four respondents, printed in full. Check that the replies are JSON and plausible.
from d2vi.synthetic import ANES2024, RespondentPrompt, parse_answer, sample_personas
ps = sample_personas(anes, 4, seed=1)
pr = RespondentPrompt(ANES2024, "political", "humanlike")
for p, r in zip(ps, gen.generate([pr.messages(p) for p in ps], seed=1)):
    print(p.describe("political")); print(r); print(parse_answer(r, ANES2024)); print("-" * 80)


In [ ]:
import json, hashlib
CONDITIONS = [("demographic", "plain"), ("demographic", "humanlike"),
              ("political", "plain"), ("political", "humanlike")]
model_idx = list(GENERATORS).index(MODEL) if MODEL in GENERATORS else 9
tables = []
for c, (level, style) in enumerate(CONDITIONS):
    seed = 20261109 + 100 * model_idx + c
    personas = sample_personas(anes, PER_CONDITION, seed=seed)
    prompt = RespondentPrompt(ANES2024, level, style)
    survey = SyntheticSurvey(ANES2024, prompt, f"{WORK}/cache_{SLUG}_{level}_{style}.csv", seed=seed)
    t = survey.run(gen, personas, prefix=f"{SLUG}-{level}-{style}")
    print(level, style, "parsed:", t.parse_ok.mean().round(3))
    tables.append(t)
synth = pd.concat(tables, ignore_index=True)
print(synth.groupby("condition").parse_ok.mean())


In [ ]:
from d2vi.features.pipeline import package_versions
ok = synth[synth.parse_ok].copy()
print(ok.groupby("condition")[[i.key for i in ANES2024.thermometers]].mean().round(1))
print("share with no open-ended text:", (ok.text.str.len() == 0).mean().round(3))
template = RespondentPrompt(ANES2024, "political", "plain").messages(sample_personas(anes, 1, seed=0)[0])
manifest = {"generator": gen.settings, "conditions": CONDITIONS, "per_condition": PER_CONDITION,
            "parsed": int(synth.parse_ok.sum()), "rows": len(synth),
            "instrument": ANES2024.name, "wording_checked": ANES2024.wording_checked,
            "prompt_sha256": hashlib.sha256(json.dumps(template).encode()).hexdigest(),
            "anes_files": [os.path.basename(src), os.path.basename(xlsx[-1])], "persona_pool": len(anes),
            "versions": package_versions("d2vi", "transformers", "torch", "bitsandbytes")}
out = f"{WORK}/anes_synth_{SLUG}.parquet"
synth.to_parquet(out, index=False)
json.dump(manifest, open(out.replace(".parquet", ".manifest.json"), "w"), indent=2)
print("saved", out)
from google.colab import files
files.download(out); files.download(out.replace(".parquet", ".manifest.json"))
